In [ ]:
#al convertir los datos a archico root tendremos trees: peak, temp (RTDs), (spectrums).
##NO FUNCIONA, MEJOR USAR UPROOT.

import ROOT
import numpy as np
import matplotlib.pyplot as plt
import cmath
import awkward as ak
import pandas as pd
import json
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from iminuit import Minuit
from iminuit.cost import UnbinnedNLL
from scipy.stats import norm
from itertools import groupby
import scipy.stats as stats
from iminuit.cost import LeastSquares
import uproot

In [ ]:
import uproot
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
import datetime
import numpy as np
import ROOT
import tqdm

In [ ]:
#fileName = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/resampled20240926.root"
#filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/resampled20240926.root"
fileName = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250227.root"
filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250227.root"
with uproot.open(fileName, xrootdsource={"chunkbytes": 1024**3}) as file:
    print(file.keys())

In [ ]:
plt.rcParams.update({
    'figure.figsize': (4*3.5, 3*3.5),
    'axes.titlesize': 35,
    'axes.titleweight': 'bold',
    'axes.titlepad': 20,  # Adjust this value to control the separation of the title from the figure
    'axes.labelsize': 25,
    'axes.labelpad': 15,  # Adjust this value to control the separation of the axes labels from the plot
    'axes.linewidth': 2.5,
    'legend.fontsize': 20,
    'legend.loc': 'best',
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'xtick.direction': 'inout',  # Ticks direction
    'ytick.direction': 'inout',
    'xtick.major.width': 2,  # Major tick width
    'ytick.major.width': 2,
    'xtick.major.size': 15,  # Major tick size
    'ytick.major.size': 15,
    'xtick.minor.width': 1,  # Minor tick width
    'ytick.minor.width': 1,
    'xtick.minor.size': 10,  # Minor tick size
    'ytick.minor.size': 10,
    'xtick.major.pad': 10,  # Padding between ticks and labels
    'ytick.major.pad': 10,
    'xtick.minor.pad': 10,
    'ytick.minor.pad': 10,
    'xtick.minor.visible': True,  # Display minor ticks
    'ytick.minor.visible': True,
    'lines.linewidth': 2.5,
    'lines.markersize': 1,
    'grid.linestyle': '--',  # Dashed grid lines
    'grid.linewidth': 0.5,
    'grid.color': 'grey',   # Grid line color
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.family': 'serif',  # Set font family to serif
    'font.serif': ['DejaVu Serif'],  # Specify a serif font, change to your preferred serif font if needed
})
plt.set_cmap('tab20')

In [ ]:
with uproot.open(fileName, xrootdsource={"chunkbytes": 1024**3}) as file:
    print(file.keys())
    #temp = file["temp"] 
    #temp = ak.to_pandas(temp.arrays())
    #temp = temp.arrays(library="pd")
    peak = file["peak"]
    #peak = peak.arrays(library="pd")
    peak = peak.arrays(library="ak")  # ak es "awkward array"

    #press = file["press"]
    #press = press.arrays(library="pd")
    #sprectrum = file["spectrum"] #puede ser temp pero no hay rama en marzo
    #spectrum = spectrum.arrays(library="pd")
    #resampled_data = file["resampled_data"] 
    #resampled_data = resampled_data.arrays(library="pd")

In [ ]:
print(resampled_data.head())
resampled_data["t"] = resampled_data["t"].apply(lambda x: datetime.datetime.utcfromtimestamp(x))
print(resampled_data.head())

In [ ]:
print(peak.head())
#peak["t[0]"] = peak["t[0]"].apply(lambda x: datetime.datetime.utcfromtimestamp(x))
#peak["t[1]"] = peak["t[1]"].apply(lambda x: datetime.datetime.utcfromtimestamp(x))
print(peak.head())

In [ ]:
import os
import pandas as pd

eos_path = "/eos/user/j/jcapotor/FBGdata/Data/pressure_setup/20250227/peaks_1.txt"

if os.path.exists(eos_path):  # Verifica si el archivo existe
    # Cargar el archivo con delimitador de espacios y sin encabezado
    df = pd.read_csv(eos_path, delim_whitespace=True, header=None, dtype=str)

    # Unir la fecha y la hora en una sola columna
    df["datetime_str"] = df[0] + " " + df[1]

    # Convertir a datetime
    df["datetime"] = pd.to_datetime(df["datetime_str"], format="%d-%b-%Y %H:%M:%S.%f", errors="coerce")

    # Convertir datetime a timestamp en segundos
    df["timestamp"] = df["datetime"].astype("int64") / 10**9  # Segundos desde Epoch

    # Eliminar columnas originales de fecha/hora y dejar solo el timestamp
    df = df.drop(columns=[0, 1, "datetime_str"])

    # Renombrar la columna timestamp como 'time' para mayor claridad
    df = df.rename(columns={"timestamp": "time"})

    print(df.head())  # Mostrar primeras filas con timestamp y otros datos
else:
    print("No se encuentra el archivo")


In [ ]:
from datetime import datetime, timedelta

def reshapeEpochTime(timestamp_ns):
    """
    Convierte un timestamp en nanosegundos a UTC, corrigiendo la diferencia GPS-UTC.
    """
    dt = datetime.utcfromtimestamp(timestamp_ns * 1e-9)  # Convertimos a segundos

    # Restamos 66 años, 9 meses y 11 días para corregir la diferencia GPS-UTC
    dt -= timedelta(days=66*365 + 30*9 + 11, hours=6)
    dt += timedelta(minutes=21)

    return dt.timestamp()

# 🔍 **Prueba con tu timestamp**
gps_timestamp = 3846334242673641341  # Número gigante del archivo
corrected_time = reshapeEpochTime(gps_timestamp)

# Convertimos a formato legible
corrected_datetime = datetime.utcfromtimestamp(corrected_time)
print(corrected_datetime)  # 🔹 Debería dar 26-Feb-2025 con la hora correcta



In [ ]:
import pandas as pd

# Cargar el archivo
eos_path = "/eos/user/j/jcapotor/FBGdata/Data/pressure_setup/20250227/peaks_1.txt"
df = pd.read_csv(eos_path, delim_whitespace=True, header=None)

# Intentar convertir la columna 2 a datetime (asumiendo que está en nanosegundos)
df["time_col2"] = pd.to_datetime(df[2] / 10**9, unit="s", errors="coerce")

# Mostrar las primeras filas con la conversión
print(df[[2, "time_col2"]].head())


In [ ]:
import ROOT

def inspect_root_file(filename):
    file = ROOT.TFile.Open(filename, "READ")
    if not file or file.IsZombie():
        print(f"Error: Cannot open file {filename}")
        return
    
    print("### File Structure ###")
    file.ls()

    trees_to_check = ["peak", "temp", "press"]
    for tree_name in trees_to_check:
        tree = file.Get(tree_name)
        if not tree:
            print(f"\nTree {tree_name} not found.")
            continue
        
        print(f"\n### Structure of {tree_name} ###")
        tree.Print()

        if tree.GetEntries() == 0:
            print(f"\nTree {tree_name} is empty.")
            continue

        print(f"\n### First 5 entries in {tree_name} ###")
        tree.GetEntry(0)  # Leer primera entrada para analizar estructura

        for entry in range(min(5, tree.GetEntries())):
            tree.GetEntry(entry)
            if tree_name == "peak":
                # Intentar determinar si wav[0] es iterable
                num_sensors = 1
                try:
                    num_sensors = len(tree.wav[0])  # Detecta cuántos sensores hay en wav
                except TypeError:
                    pass  # Si no tiene len(), significa que es un solo valor (float)

                wav_values_s = [tree.wav[0][i] for i in range(num_sensors)] if num_sensors > 1 else [tree.wav[0]]
                wav_values_p = [tree.wav[1][i] for i in range(num_sensors)] if num_sensors > 1 else [tree.wav[1]]
                
                print(f"Entry {entry}: t_s={tree.t[0]}, t_p={tree.t[1]}, wav_s={wav_values_s}, wav_p={wav_values_p} (sensors={num_sensors})")

            elif tree_name == "temp":
                temp_values = [tree.temp[i] for i in range(len(tree.temp))] if hasattr(tree, 'temp') else []
                print(f"Entry {entry}: t={tree.t}, temp={temp_values}")

            elif tree_name == "press":
                press_value = tree.press if hasattr(tree, 'press') else "N/A"
                print(f"Entry {entry}: t={tree.t}, press={press_value}")

    file.Close()

# Llamar la función con un archivo de prueba
inspect_root_file(filename)





In [ ]:
##codigo preeliminar para resamplear que falla por el numero de count entries q no es igual y no tenemos un solo tree con 4 branches, una de 't' igual para todos y la de 'wav', 'temp' y 'press'
import ROOT
import numpy as np

def resample_root_file(input_filename, output_filename, sampling_interval=5, default_value=-99999):
    """
    Resamples the trees ('peak', 'temp', 'press') in a ROOT file.
    It defines a global time t0 as the minimum across the trees and averages
    all entries within each 5-second interval. If no data is present in an interval,
    a default value (e.g., -999) is assigned.
    """
    file = ROOT.TFile.Open(input_filename, "READ")
    if not file or file.IsZombie():
        print(f"Error: Cannot open file {input_filename}")
        return

    output_file = ROOT.TFile(output_filename, "RECREATE")
    trees_to_resample = ['peak', 'temp', 'press']
    # Step 1: Find the global initial time t0
    t0 = float('inf')
    for tree_name in trees_to_resample:
        tree = file.Get(tree_name)
        if tree and tree.GetEntries() > 0:
            tree.GetEntry(0)  # Get the first entry
            if tree_name == 'peak':
                min_time = min(tree.t[0], tree.t[1])  # Extract both polarization times
            else:
                min_time = tree.t  # Extract the first entry time for 'temp' and 'press'
            t0 = min(t0, min_time)  # Update global minimum time
    print(f"Global initial time t0: {t0}")
    
    # Step 2: Process each tree and apply resampling
    for tree_name in trees_to_resample:
        tree = file.Get(tree_name)
        if not tree:
            continue
        # Read time and all branches dynamically
        branches = {}
        time_values = []
        count_entries = []
        
        tree.GetEntry(0)  # Obtain the first entry to determine dimensions
        if tree_name == 'peak':
            if hasattr(tree, 'wav') and isinstance(tree.wav[0], ROOT.vector('double')):
                num_sensors = len(tree.wav[0])  # Determine the number of sensors in the fiber
            else:
                num_sensors = 1  # Assume a single sensor if not iterable
            print(f"Detected {num_sensors} sensors in wav")
        
        for entry in range(tree.GetEntries()):
            tree.GetEntry(entry)
            count_entries.append(1)
            if tree_name == 'peak':
                time_values.append(min(tree.t[0], tree.t[1]))
                for sensor in range(num_sensors):
                    branches.setdefault(f'wav_s_{sensor}', []).append(tree.wav[0][sensor] if num_sensors > 1 else tree.wav[0])
                    branches.setdefault(f'wav_p_{sensor}', []).append(tree.wav[1][sensor] if num_sensors > 1 else tree.wav[1])
            elif tree_name == 'temp':
                time_values.append(tree.t)
                for i in range(8):
                    branches.setdefault(f'temp_{i}', []).append(tree.temp[i])
            elif tree_name == 'press':
                time_values.append(tree.t)
                branches.setdefault('press', []).append(tree.press)
        
        time_values = np.array(time_values)
        count_entries = np.array(count_entries)
        for branch_name in branches:
            branches[branch_name] = np.array(branches[branch_name])
        
        max_t = np.max(time_values)
        bins = np.arange(t0, max_t + sampling_interval, sampling_interval)
        resampled_data = {b: {name: default_value for name in branches} for b in bins[:-1]}
        std_data = {b: {name: 0 for name in branches} for b in bins[:-1]}
        count_data = {b: 0 for b in bins[:-1]}
        
        for i in range(len(bins) - 1):
            mask = (time_values >= bins[i]) & (time_values < bins[i + 1])
            count_data[bins[i]] = np.sum(mask)
            if np.any(mask):
                for branch_name in branches:
                    values = branches[branch_name][mask]
                    resampled_data[bins[i]][branch_name] = np.mean(values)
                    std_data[bins[i]][branch_name] = np.std(values)
        
        resampled_tree = ROOT.TTree(f"{tree_name}_resample", f"Resampled {tree_name}")
        resampled_t = np.zeros(1, dtype=np.float64)
        resampled_branches = {name: np.zeros(1, dtype=np.float64) for name in branches}
        resampled_std = {name: np.zeros(1, dtype=np.float64) for name in branches}
        resampled_count = np.zeros(1, dtype=np.int32)
        
        t_branch = resampled_tree.Branch("t", resampled_t, "t/D")
        count_branch = resampled_tree.Branch("count_entries", resampled_count, "count_entries/I")
        data_branches = {name: resampled_tree.Branch(name, resampled_branches[name], f"{name}/D") for name in branches}
        std_branches = {name: resampled_tree.Branch(f"{name}_std", resampled_std[name], f"{name}_std/D") for name in branches}
        
        for i, b in enumerate(bins[:-1]):
            resampled_t[0] = b
            resampled_count[0] = count_data[b]
            for name in branches:
                resampled_branches[name][0] = resampled_data[b][name]
                resampled_std[name][0] = std_data[b][name]
            resampled_tree.Fill()
        
        resampled_tree.Write()
    
    output_file.Close()
    file.Close()
    print(f"Resampling completed. Output file: {output_filename}")

resample_root_file("/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20240926.root", "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/resampled20240926.root", sampling_interval=5, default_value=-99999)

In [ ]:
import ROOT
import numpy as np

##hace bien la estructura pero no calcula la media y tarda demasiado en correr, se modifica en visual studio

def resample_root_file(input_filename, output_filename, sampling_interval=5, default_value=-99999):
    """
    Resamples the trees ('peak', 'temp', 'press') in a ROOT file.
    Creates a single tree containing time, wav, temp, and press arrays.
    """
    file = ROOT.TFile.Open(input_filename, "READ")
    if not file or file.IsZombie():
        print(f"Error: Cannot open file {input_filename}")
        return

    output_file = ROOT.TFile(output_filename, "RECREATE")
    trees_to_resample = ['peak', 'temp', 'press']
    
    # Step 1: Find the global initial time t0
    t0 = float('inf')
    max_t = float('-inf')
    num_sensors = None
    num_polarizations = None
    
    for tree_name in trees_to_resample:
        tree = file.Get(tree_name)
        if tree and tree.GetEntries() > 0:
            tree.GetEntry(0)
            min_time = min(tree.t[0], tree.t[1]) if tree_name == 'peak' else tree.t
            t0 = min(t0, min_time)
            
            tree.GetEntry(tree.GetEntries() - 1)
            max_time = max(tree.t[0], tree.t[1]) if tree_name == 'peak' else tree.t
            max_t = max(max_t, max_time)
            
            if tree_name == 'peak' and hasattr(tree, 'wav'):
                num_polarizations = len(tree.wav)
                num_sensors = len(tree.wav[0]) if num_polarizations > 0 else 1
    
    if num_sensors is None or num_polarizations is None:
        print("Error: Could not determine wav dimensions.")
        return

    print(f"Global initial time t0: {t0}")
    bins = np.arange(t0, max_t + sampling_interval, sampling_interval)
    
    # Step 2: Prepare data storage
    resampled_data = {b: {
        "wav": np.full((num_polarizations, num_sensors), default_value), 
        "temp": np.full(8, default_value), 
        "press": default_value
    } for b in bins[:-1]}
    count_data = {b: 0 for b in bins[:-1]}
    
    for tree_name in trees_to_resample:
        tree = file.Get(tree_name)
        if not tree:
            continue
        
        for entry in range(tree.GetEntries()):
            tree.GetEntry(entry)
            time_value = min(tree.t[0], tree.t[1]) if tree_name == 'peak' else tree.t
            
            for i in range(len(bins) - 1):
                if bins[i] <= time_value < bins[i + 1]:
                    count_data[bins[i]] += 1
                    if tree_name == 'peak':
                        resampled_data[bins[i]]["wav"] = np.array(tree.wav) if hasattr(tree, 'wav') else np.full((num_polarizations, num_sensors), default_value)
                    elif tree_name == 'temp':
                        resampled_data[bins[i]]["temp"] = np.array(tree.temp) if hasattr(tree, 'temp') else np.full(8, default_value)
                    elif tree_name == 'press':
                        resampled_data[bins[i]]["press"] = tree.press if hasattr(tree, 'press') else default_value
                    break
    
    # Step 3: Create a single resampled tree
    resampled_tree = ROOT.TTree("resampled_data", "Resampled Data")
    resampled_t = np.zeros(1, dtype=np.float64)
    resampled_count = np.zeros(1, dtype=np.int32)
    resampled_wav = np.zeros((num_polarizations, num_sensors), dtype=np.float64)
    resampled_temp = np.zeros(8, dtype=np.float64)
    resampled_press = np.zeros(1, dtype=np.float64)
    
    resampled_tree.Branch("t", resampled_t, "t/D")
    resampled_tree.Branch("count_entries", resampled_count, "count_entries/I")
    resampled_tree.Branch("wav", resampled_wav, f"wav[{num_polarizations}][{num_sensors}]/D")
    resampled_tree.Branch("temp", resampled_temp, "temp[8]/D")
    resampled_tree.Branch("press", resampled_press, "press/D")
    
    for i, b in enumerate(bins[:-1]):
        resampled_t[0] = b
        resampled_count[0] = count_data[b]
        resampled_wav[:] = resampled_data[b]["wav"]
        resampled_temp[:] = resampled_data[b]["temp"]
        resampled_press[0] = resampled_data[b]["press"]
        resampled_tree.Fill()
    
    resampled_tree.Write()
    output_file.Close()
    file.Close()
    print(f"Resampling completed. Output file: {output_filename}")

resample_root_file("/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250227.root", "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/resampled20250227.root", sampling_interval=5, default_value=-99999)

In [ ]:
Initial time for peak: 1637341842.673642
Max time for peak: 1637370436.691008
Structure of wav: 2 polarizations, 3 sensors
Initial time for temp: 1740655797.0
Max time for temp: 1740687182.0
Initial time for press: 1740656050.0
Max time for press: 1740687250.0
Trees being processed: ['peak', 'temp', 'press']
Global initial time t0: 1637341842.673642
Number of intervals: 20669082

In [ ]:
import ROOT
import numpy as np

def resample_root_file(input_filename, output_filename, sampling_interval=5, default_value=-99999):
    """
    Resamples the trees ('peak', 'temp', 'press') in a ROOT file.
    Creates a single tree containing time, wav, temp, and press arrays.
    """
    file = ROOT.TFile.Open(input_filename, "READ")
    if not file or file.IsZombie():
        print(f"Error: Cannot open file {input_filename}")
        return

    output_file = ROOT.TFile(output_filename, "RECREATE")
    trees_to_resample = ['peak', 'temp', 'press']
    
    # Step 1: Find the global initial time t0
    t0 = float('inf')
    max_t = float('-inf')
    num_sensors = None
    num_polarizations = 2  # Always defined as 2
    
    processed_trees = []
    
    for tree_name in trees_to_resample:
        tree = file.Get(tree_name)
        if tree and tree.GetEntries() > 0:
            processed_trees.append(tree_name)
            tree.GetEntry(0)
            min_time = min(tree.t[0], tree.t[1]) if tree_name == 'peak' else tree.t
            t0 = min(t0, min_time)
            
            tree.GetEntry(tree.GetEntries() - 1)
            max_time = max(tree.t[0], tree.t[1]) if tree_name == 'peak' else tree.t
            max_t = max(max_t, max_time)
            
            if tree_name == 'peak':
                if hasattr(tree, 'wav'):
                    num_sensors = len(tree.wav[0]) if len(tree.wav) > 0 else 1
                    print(f"Structure of wav: {num_polarizations} polarizations, {num_sensors} sensors")
                else:
                    print("Warning: 'peak' tree does not contain 'wav' attribute.")
    
    print(f"Trees being processed: {processed_trees}")
    
    if num_sensors is None:
        print("Error: Could not determine wav dimensions.")
        return

    print(f"Global initial time t0: {t0}")
    bins = np.arange(t0, max_t + sampling_interval, sampling_interval)
    
    # Step 2: Prepare data storage
    resampled_data = {b: {"wav": [], "temp": [], "press": []} for b in bins[:-1]}
    count_temp = {b: 0 for b in bins[:-1]}
    count_press = {b: 0 for b in bins[:-1]}
    count_wav = {b: 0 for b in bins[:-1]}
    
    for tree_name in trees_to_resample:
        tree = file.Get(tree_name)
        if not tree:
            continue
        
        for entry in range(tree.GetEntries()):
            tree.GetEntry(entry)
            time_value = min(tree.t[0], tree.t[1]) if tree_name == 'peak' else tree.t
            
            idx = np.searchsorted(bins, time_value, side='right') - 1
            if 0 <= idx < len(bins) - 1:
                if tree_name == 'peak' and hasattr(tree, 'wav'):
                    resampled_data[bins[idx]]["wav"].append(np.array(tree.wav))
                    count_wav[bins[idx]] += 1
                elif tree_name == 'temp' and hasattr(tree, 'temp'):
                    resampled_data[bins[idx]]["temp"].append(np.array(tree.temp))
                    count_temp[bins[idx]] += 1
                elif tree_name == 'press' and hasattr(tree, 'press'):
                    resampled_data[bins[idx]]["press"].append(tree.press)
                    count_press[bins[idx]] += 1
    
    # Step 3: Compute mean values ignoring default values
    for b in bins[:-1]:
        if count_wav[b] > 0:
            wav_values = np.array(resampled_data[b]["wav"])
            wav_values = np.where(wav_values == default_value, np.nan, wav_values)
            if np.all(np.isnan(wav_values)):
                resampled_data[b]["wav"] = np.full((num_polarizations, num_sensors), default_value)
            else:
                resampled_data[b]["wav"] = np.nanmean(wav_values, axis=0)
        else:
            resampled_data[b]["wav"] = np.full((num_polarizations, num_sensors), default_value)
        
        if count_temp[b] > 0:
            temp_values = np.array(resampled_data[b]["temp"])
            temp_values = np.where(temp_values == default_value, np.nan, temp_values)
            if np.all(np.isnan(temp_values)):
                print(f"Interval {b}: No valid temp data found, setting to default value.")
                resampled_data[b]["temp"] = np.full(8, default_value)
            else:
                resampled_data[b]["temp"] = np.nanmean(temp_values, axis=0)
        else:
            print(f"Interval {b}: No valid temp data found, setting to default value.")
            resampled_data[b]["temp"] = np.full(8, default_value)
        
        if count_press[b] > 0:
            press_values = np.array(resampled_data[b]["press"])
            press_values = np.where(press_values == default_value, np.nan, press_values)
            if np.all(np.isnan(press_values)):
                resampled_data[b]["press"] = default_value
            else:
                resampled_data[b]["press"] = np.nanmean(press_values)
        else:
            resampled_data[b]["press"] = default_value
    
    # Step 4: Create a single resampled tree
    resampled_tree = ROOT.TTree("resampled_data", "Resampled Data")
    resampled_t = np.zeros(1, dtype=np.float64)
    resampled_wav = np.zeros((num_polarizations, num_sensors), dtype=np.float64)
    resampled_temp = np.zeros(8, dtype=np.float64)
    resampled_press = np.zeros(1, dtype=np.float64)
    
    resampled_tree.Branch("t", resampled_t, "t/D")
    resampled_tree.Branch("wav", resampled_wav, f"wav[{num_polarizations}][{num_sensors}]/D")
    resampled_tree.Branch("temp", resampled_temp, "temp[8]/D")
    resampled_tree.Branch("press", resampled_press, "press/D")
    
    for i, b in enumerate(bins[:-1]):
        resampled_t[0] = b
        resampled_wav[:] = resampled_data[b]["wav"]
        resampled_temp[:] = resampled_data[b]["temp"]
        resampled_press[0] = resampled_data[b]["press"]
        resampled_tree.Fill()
    
    resampled_tree.Write()
    output_file.Close()
    file.Close()
    print(f"Resampling completed. Output file: {output_filename}")

resample_root_file("/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250227.root", "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/resampled20250227.root", sampling_interval=5, default_value=-99999)


In [ ]:
#ultima version buena subida a github que da demasiados valores en -999
import ROOT
import numpy as np

def resample_root_file(input_filename, output_filename, sampling_interval=10, default_value=-99999):
    """
    Resamples the trees ('peak', 'temp', 'press') in a ROOT file.
    Creates a single tree containing time, wav, temp, and press arrays.
    """
    file = ROOT.TFile.Open(input_filename, "READ")
    if not file or file.IsZombie():
        print(f"Error: Cannot open file {input_filename}")
        return

    output_file = ROOT.TFile(output_filename, "RECREATE")
    trees_to_resample = ['peak', 'temp', 'press']
    
    # Step 1: Find the global initial time t0
    t0 = float('inf')
    max_t = float('-inf')
    num_sensors = None
    num_polarizations = 2  # Always defined as 2
    
    processed_trees = []
    
    for tree_name in trees_to_resample:
        tree = file.Get(tree_name)
        if tree and tree.GetEntries() > 0:
            processed_trees.append(tree_name)
            tree.GetEntry(0)
            min_time = min(tree.t[0], tree.t[1]) if tree_name == 'peak' else tree.t
            t0 = min(t0, min_time)
            print(f"{tree_name} initial time: {min_time}")
            
            tree.GetEntry(tree.GetEntries() - 1)
            max_time = max(tree.t[0], tree.t[1]) if tree_name == 'peak' else tree.t
            max_t = max(max_t, max_time)
            print(f"{tree_name} final time: {max_time}")
            
            if tree_name == 'peak':
                if hasattr(tree, 'wav'):
                    num_sensors = len(tree.wav[0]) if len(tree.wav) > 0 else 1
                    print(f"Structure of wav: {num_polarizations} polarizations, {num_sensors} sensors")
                else:
                    print("Warning: 'peak' tree does not contain 'wav' attribute.")
    
    print(f"Trees being processed: {processed_trees}")
    
    if num_sensors is None:
        print("Error: Could not determine wav dimensions.")
        return

    print(f"Global initial time t0: {t0}")
    print(f"Global final time max_t: {max_t}")
    bins = np.arange(t0, max_t + sampling_interval, sampling_interval)
    print(f"Number of intervals: {len(bins) - 1}")
    
    # Step 2: Prepare data storage
    resampled_data = {b: {"wav": [], "temp": [], "press": []} for b in bins[:-1]}
    count_temp = {b: 0 for b in bins[:-1]}
    count_press = {b: 0 for b in bins[:-1]}
    count_wav = {b: 0 for b in bins[:-1]}
    
    for tree_name in trees_to_resample:
        tree = file.Get(tree_name)
        if not tree:
            continue
        
        for entry in range(tree.GetEntries()):
            tree.GetEntry(entry)
            time_value = min(tree.t[0], tree.t[1]) if tree_name == 'peak' else tree.t
            
            idx = np.searchsorted(bins, time_value, side='right') - 1
            if 0 <= idx < len(bins) - 1:
                if tree_name == 'peak' and hasattr(tree, 'wav'):
                    resampled_data[bins[idx]]["wav"].append(np.array(tree.wav))
                    count_wav[bins[idx]] += 1
                elif tree_name == 'temp' and hasattr(tree, 'temp'):
                    resampled_data[bins[idx]]["temp"].append(np.array(tree.temp))
                    count_temp[bins[idx]] += 1
                elif tree_name == 'press' and hasattr(tree, 'press'):
                    resampled_data[bins[idx]]["press"].append(tree.press)
                    count_press[bins[idx]] += 1
    
    # Step 3: Compute mean values ignoring default values
    for b in bins[:-1]:
        if count_wav[b] > 0:
            wav_values = np.array(resampled_data[b]["wav"])
            wav_values = np.where(wav_values == default_value, np.nan, wav_values)
            if np.all(np.isnan(wav_values)):
                resampled_data[b]["wav"] = np.full((num_polarizations, num_sensors), default_value)
            else:
                resampled_data[b]["wav"] = np.nanmean(wav_values, axis=0)
        else:
            resampled_data[b]["wav"] = np.full((num_polarizations, num_sensors), default_value)
        
        if count_temp[b] > 0:
            temp_values = np.array(resampled_data[b]["temp"])
            temp_values = np.where(temp_values == default_value, np.nan, temp_values)
            if np.all(np.isnan(temp_values)):
                resampled_data[b]["temp"] = np.full(8, default_value)
            else:
                resampled_data[b]["temp"] = np.nanmean(temp_values, axis=0)
        else:
            resampled_data[b]["temp"] = np.full(8, default_value)
        
        if count_press[b] > 0:
            press_values = np.array(resampled_data[b]["press"])
            press_values = np.where(press_values == default_value, np.nan, press_values)
            if np.all(np.isnan(press_values)):
                resampled_data[b]["press"] = default_value
            else:
                resampled_data[b]["press"] = np.nanmean(press_values)
        else:
            resampled_data[b]["press"] = default_value
    
    # Step 4: Create a single resampled tree
    resampled_tree = ROOT.TTree("resampled_data", "Resampled Data")
    resampled_t = np.zeros(1, dtype=np.float64)
    resampled_wav = np.zeros((num_polarizations, num_sensors), dtype=np.float64)
    resampled_temp = np.zeros(8, dtype=np.float64)
    resampled_press = np.zeros(1, dtype=np.float64)
    
    resampled_tree.Branch("t", resampled_t, "t/D")
    resampled_tree.Branch("wav", resampled_wav, f"wav[{num_polarizations}][{num_sensors}]/D")
    resampled_tree.Branch("temp", resampled_temp, "temp[8]/D")
    resampled_tree.Branch("press", resampled_press, "press/D")
    
    for i, b in enumerate(bins[:-1]):
        resampled_t[0] = b
        resampled_wav[:] = resampled_data[b]["wav"]
        resampled_temp[:] = resampled_data[b]["temp"]
        resampled_press[0] = resampled_data[b]["press"]
        resampled_tree.Fill()
    
    resampled_tree.Write()
    output_file.Close()
    file.Close()
    print(f"Resampling completed. Output file: {output_filename}")

resample_root_file("/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250228.root", "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/resampled20250228.root", sampling_interval=30, default_value=-9999)

In [ ]:
#version que funciona y aun no he subido a github

import ROOT
import numpy as np
import datetime

def resample_root_file(input_filename, output_filename, sampling_interval=5, default_value=-99999):
    """
    Resamples the trees ('peak', 'temp', 'press') in a ROOT file.
    Creates a single tree containing time, wav, temp, and press arrays.
    """
    file = ROOT.TFile.Open(input_filename, "READ")
    if not file or file.IsZombie():
        print(f"Error: Cannot open file {input_filename}")
        return

    output_file = ROOT.TFile(output_filename, "RECREATE")
    trees_to_resample = ['peak', 'temp', 'press']
    
    # Step 1: Find the global initial time t0
    t0 = float('inf')
    max_t = float('-inf')
    num_sensors = None
    num_polarizations = 2  # Always defined as 2
    
    processed_trees = []
    
    for tree_name in trees_to_resample:
        tree = file.Get(tree_name)
        if tree and tree.GetEntries() > 0:
            processed_trees.append(tree_name)
            tree.GetEntry(0)
            min_time = min(tree.t[0], tree.t[1]) if tree_name == 'peak' else tree.t
            t0 = min(t0, min_time)
            #print(f"{tree_name} initial time: {min_time}")
            
            tree.GetEntry(tree.GetEntries() - 1)
            max_time = max(tree.t[0], tree.t[1]) if tree_name == 'peak' else tree.t
            max_t = max(max_t, max_time)
            #print(f"{tree_name} final time: {max_time}")
            print(f"{tree_name} initial time: {min_time} ({datetime.datetime.utcfromtimestamp(min_time)})")
            print(f"{tree_name} final time: {max_time} ({datetime.datetime.utcfromtimestamp(max_time)})")

            
            if tree_name == 'peak':
                if hasattr(tree, 'wav'):
                    num_sensors = len(tree.wav[0]) if len(tree.wav) > 0 else 1
                    print(f"Structure of wav: {num_polarizations} polarizations, {num_sensors} sensors")
                else:
                    print("Warning: 'peak' tree does not contain 'wav' attribute.")
    
    print(f"Trees being processed: {processed_trees}")
    
    if num_sensors is None:
        print("Error: Could not determine wav dimensions.")
        return

    #print(f"Global initial time t0: {t0}")
    #print(f"Global final time max_t: {max_t}")
    print(f"Global initial time t0: {t0} ({datetime.datetime.utcfromtimestamp(t0)})")
    print(f"Global final time max_t: {max_t} ({datetime.datetime.utcfromtimestamp(max_t)})")

    bins = np.arange(t0, max_t + sampling_interval, sampling_interval)
    print(f"Number of intervals: {len(bins) - 1}")
    
    # Step 2: Prepare data storage
    resampled_data = {b: {"wav": [], "temp": [], "press": []} for b in bins[:-1]}
    count_temp = {b: 0 for b in bins[:-1]}
    count_press = {b: 0 for b in bins[:-1]}
    count_wav = {b: 0 for b in bins[:-1]}
    
    for tree_name in trees_to_resample:
        tree = file.Get(tree_name)
        if not tree:
            continue
        
        for entry in range(tree.GetEntries()):
            tree.GetEntry(entry)
            time_value = min(tree.t[0], tree.t[1]) if tree_name == 'peak' else tree.t
            
            idx = np.searchsorted(bins, time_value, side='right') - 1
            if 0 <= idx < len(bins) - 1:
                if tree_name == 'peak' and hasattr(tree, 'wav'):
                    resampled_data[bins[idx]]["wav"].append(np.array(tree.wav))
                    count_wav[bins[idx]] += 1
                elif tree_name == 'temp' and hasattr(tree, 'temp'):
                    resampled_data[bins[idx]]["temp"].append(np.array(tree.temp))
                    count_temp[bins[idx]] += 1
                elif tree_name == 'press' and hasattr(tree, 'press'):
                    resampled_data[bins[idx]]["press"].append(tree.press)
                    count_press[bins[idx]] += 1
    
    # Step 3: Compute mean values ignoring default values
    for b in bins[:-1]:
        if count_wav[b] == 0 and count_temp[b] == 0 and count_press[b] == 0:
            start_time = datetime.datetime.utcfromtimestamp(b)
            end_time = datetime.datetime.utcfromtimestamp(b + sampling_interval)
            print(f"Interval {b}-{b+sampling_interval} ({start_time} - {end_time}) contains no valid data.")
            #print(f"Interval {b}-{b+sampling_interval} contains no valid data.")
        
        if count_wav[b] > 0:
            wav_values = np.array(resampled_data[b]["wav"])
            wav_values = np.where(wav_values == default_value, np.nan, wav_values)
            resampled_data[b]["wav"] = np.nanmean(wav_values, axis=0) if not np.all(np.isnan(wav_values)) else np.full((num_polarizations, num_sensors), default_value)
        else:
            resampled_data[b]["wav"] = np.full((num_polarizations, num_sensors), default_value)
        
        if count_temp[b] > 0:
            temp_values = np.array(resampled_data[b]["temp"])
            temp_values = np.where(temp_values == default_value, np.nan, temp_values)
            resampled_data[b]["temp"] = np.nanmean(temp_values, axis=0) if not np.all(np.isnan(temp_values)) else np.full(8, default_value)
        else:
            resampled_data[b]["temp"] = np.full(8, default_value)
        
        if count_press[b] > 0:
            press_values = np.array(resampled_data[b]["press"])
            press_values = np.where(press_values == default_value, np.nan, press_values)
            resampled_data[b]["press"] = np.nanmean(press_values) if not np.all(np.isnan(press_values)) else default_value
        else:
            resampled_data[b]["press"] = default_value
    
    # Step 4: Create a single resampled tree
    resampled_tree = ROOT.TTree("resampled_data", "Resampled Data")
    resampled_t = np.zeros(1, dtype=np.float64)
    resampled_wav = np.zeros((num_polarizations, num_sensors), dtype=np.float64)
    resampled_temp = np.zeros(8, dtype=np.float64)
    resampled_press = np.zeros(1, dtype=np.float64)
    
    resampled_tree.Branch("t", resampled_t, "t/D")
    resampled_tree.Branch("wav", resampled_wav, f"wav[{num_polarizations}][{num_sensors}]/D")
    resampled_tree.Branch("temp", resampled_temp, "temp[8]/D")
    resampled_tree.Branch("press", resampled_press, "press/D")
    
    for i, b in enumerate(bins[:-1]):
        resampled_t[0] = b + sampling_interval / 2  # Use center of interval
        resampled_wav[:] = resampled_data[b]["wav"]
        resampled_temp[:] = resampled_data[b]["temp"]
        resampled_press[0] = resampled_data[b]["press"]
        resampled_tree.Fill()
    
    resampled_tree.Write()
    output_file.Close()
    file.Close()
    print(f"Resampling completed. Output file: {output_filename}")


resample_root_file("/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20241001.root", "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/resampled20241001.root", sampling_interval=30, default_value=-9999)

In [ ]:
outputFile = ROOT.TFile(fileName, "READ")

# Obtener el árbol correspondiente
temp_tree = outputFile.Get("temp")
peak_tree = outputFile.Get("peak")
#spectrum_tree = outputFile.Get("spectrum")

In [ ]:
#listas para almacenar los valores

t_peak = []
wav_values_cc_0 = {"p": [], "s": []}
t_values_temp = []
temp_values = []


# Iterar sobre las entradas del árbol peak
for entry in peak_tree:
    wav_values = np.array(entry.wav).reshape(2, 1) #dos polarizaciones y 1 canales o sensores ??
    tt_peak = np.array(entry.t).reshape(2)
    t_peak.append(tt_peak)
    wav_values_cc_0['p'].append(wav_values[0][0]) #cojo el primer sensor solo [0] pero el codigo solo me saca la pol. p 
    wav_values_cc_0['s'].append(wav_values[1][0])

for entry in temp_tree:
    t_ = entry.t
    temp_ = entry.temp
    t_values_temp.append(t_)
    temp_values.append(temp_values)
    
# Convertir las listas en arrays NumPy
t_values_temp = np.array(t_values_temp)
temp_values = np.array(temp_values)
wav_values_cc_0 = {k: np.array(v) for k, v in wav_values_cc_0.items()}
t_peak = np.array(t_peak)


In [ ]:
def findPlateaus(temp_tree, branch="temp", tolerance = 0.004, min_plateau_length = 50): #cambiar temp si no hay esa rama 
    nEntries = temp_tree.GetEntries()
    PLATEAU_TIMES = {}  # Dictionary to store start and end timestamps for each plateau
    temp_tree.GetEntry(0)
    VALUE0, T0 = getattr(temp_tree, branch), getattr(temp_tree, "t")
    print(VALUE0)
    nPlateaus = 0
    for i in range(1, nEntries):
        temp_tree.GetEntry(i)
        VALUE, T = getattr(temp_tree, branch), getattr(temp_tree, "t")

        # Check if the current value is close to the previous value
        if abs(VALUE - VALUE0) < tolerance:
            continue
        else:
            if abs(T-T0) < min_plateau_length:
                print(VALUE0, VALUE)
                T0 = T
                VALUE0 = VALUE
                continue
            else:
                PLATEAU_TIMES[nPlateaus] = [(T0, T)]
                nPlateaus += 1
                T0 = T
                VALUE0 = VALUE

    return PLATEAU_TIMES

plateau_regions = findPlateaus(temp_tree, branch="temp", tolerance=0.5, min_plateau_length=60*25) #busco los plateaus en temp y no en los sensores (ver plots de control)
